# PACE Phase 11B — Independent QMAP MIC / HC50 Benchmark (analysis only)

**Goal:** Compare PACE's 90 sequence-descriptor representation to a 21-feature composition baseline on predefined QMAP *E. coli* MIC and mammalian HC50 test splits. Also test the original PACE hemolysis training source against held-out QMAP HC50 (homology filtering applied).

**Scientific scope:** This is a supplementary external **regression generalization** benchmark, **not** a comparison of PACE's six ESKAPE-specific MIC models, not a peptide efficacy study, and not a claim of generator superiority. QMAP includes sequence modifications not represented by PACE's 90D features; we therefore restrict *both* training and evaluation to canonical, all-L, unmodified-terminal sequences of length 5–60, and report the restricted subset sizes. The resulting metrics must **not** be compared directly with QMAP's unrestricted leaderboard.

No manuscript, supplied supplementary file, or previous result will be modified. The source file SHA256 is retained in the output.

**You will need:** only `PACE_Phase8_ESM2_results.zip` from the earlier successful Colab experiment. The original PACE feature code is extracted from it. All QMAP data are distributed through the package. The optional original PACE hemolysis CSV is downloaded from the published repository.

In [ ]:
%pip -q install 'qmap-benchmark' 'biopython>=1.81' 'scikit-learn>=1.3' 'pandas>=2' 'numpy>=1.24' requests
from qmap import QMAPBenchmark, DBAASPDataset
from sklearn.ensemble import ExtraTreesRegressor
import numpy as np, pandas as pd
print('QMAP and scientific packages imported successfully')

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import json, csv, io, hashlib, importlib.util, sys, time, platform
ROOT=Path('/content');OUT=ROOT/'PACE_Phase11B_QMAP_outputs';OUT.mkdir(exist_ok=True)
zpath=ROOT/'PACE_Phase8_ESM2_results.zip'
if not zpath.exists():
    print('Upload PACE_Phase8_ESM2_results.zip from your earlier Colab results')
    uploaded=files.upload()
    for name, content in uploaded.items(): (ROOT/Path(name).name).write_bytes(content)
    zpath=ROOT/'PACE_Phase8_ESM2_results.zip'
assert zpath.exists(),'Please upload PACE_Phase8_ESM2_results.zip'
with ZipFile(zpath) as z:
    code=z.read('pace_original_features.py')
    manifest=json.loads(z.read('manifest.json'))
feature_src=OUT/'pace_original_features.py';feature_src.write_bytes(code)
spec=importlib.util.spec_from_file_location('pace_qmap_features',feature_src)
feat=importlib.util.module_from_spec(spec);sys.modules[spec.name]=feat;spec.loader.exec_module(feat)
assert len(feat.FEATURE_NAMES)==90, len(feat.FEATURE_NAMES)
print('Verified 90D PACE descriptor code SHA256:',hashlib.sha256(code).hexdigest())
print('Previous Colab manifest available:',bool(manifest))

In [ ]:
from functools import lru_cache
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from scipy.stats import pearsonr,spearmanr
AA='ACDEFGHIKLMNPQRSTVWY';AASET=set(AA)

@lru_cache(maxsize=100000)
def features90(s):
    a=np.asarray(feat.featurize(s),dtype=float)
    assert a.shape==(90,) and np.isfinite(a).all(),s
    return a
@lru_cache(maxsize=100000)
def features21(s):
    return np.asarray([len(s)]+[s.count(a)/len(s) for a in AA],dtype=float)

def get_X(seqs, dim):
    fn=features90 if dim==90 else features21
    return np.stack([fn(str(s)) for s in seqs],axis=0)

def model_fit_predict(tr_seq,tr_log10,te_seq,dim):
    m=ExtraTreesRegressor(n_estimators=150,random_state=2024,n_jobs=-1,
                          min_samples_leaf=1,max_features=1.0)
    m.fit(get_X(tr_seq,dim),np.asarray(tr_log10,dtype=float))
    return m.predict(get_X(te_seq,dim))

def canonical_subset(dataset):
    d=(dataset.with_canonical_only()
              .with_l_aa_only()
              .with_length_range(5,60)
              .with_terminal_modification(False,False))
    # Dataset APIs allow cases with bonds/disulfides; their representations are
    # not explicitly encoded by PACE descriptors; we report this limitation.
    return d

def metric_row(y,p):
    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
    return dict(n=len(y),r2=float(r2_score(y,p)),
         rmse=float(np.sqrt(mean_squared_error(y,p))),
         mae=float(mean_absolute_error(y,p)),
         pearson=float(pearsonr(y,p).statistic),
         spearman=float(spearmanr(y,p).statistic))

runs=[];preds=[];errors=[]
DATASET=DBAASPDataset()
print('QMAP dataset available; source library sequences:',len(DATASET.sequences))

## MIC task: independent *E. coli* QMAP evaluation

Original PACE does **not** train an *E. coli* MIC endpoint. This is a **new model using PACE feature engineering**, fitted to QMAP's permitted DBAASP training sequences. It cannot be presented as an external validation of the original six ESKAPE models. Homology masks prevent sequences too similar to each QMAP test split from entering training.

In [ ]:
SPECIES='Escherichia coli'
dbase=canonical_subset(DATASET.with_bacterial_targets([SPECIES]))
assert len(dbase)>0
for split in range(5):
    try:
        benchmark=canonical_subset(QMAPBenchmark(split=split).with_bacterial_targets([SPECIES]))
        trmask=benchmark.get_train_mask(list(dbase.sequences),threshold=.6,show_progress=False,num_threads=2)
        train=dbase[trmask]
        seqs=list(train.sequences);testseq=list(benchmark.sequences)
        ys=np.array([float(x.targets[SPECIES].consensus) for x in train],dtype=float)
        yt=np.array([float(x.targets[SPECIES].consensus) for x in benchmark],dtype=float)
        keep=np.isfinite(ys)&(ys>0)
        ok=np.isfinite(yt)&(yt>0)
        seqs=[s for s,v in zip(seqs,keep) if v]
        testseq=[s for s,v in zip(testseq,ok) if v]
        ys=np.log10(ys[keep]);yt=np.log10(yt[ok])
        if len(seqs)<40 or len(testseq)<20: raise RuntimeError('Not enough train/test samples after filters')
        for dim in (21,90):
            yp=model_fit_predict(seqs,ys,testseq,dim)
            row={'task':'MIC_E_coli','split':split,'features':dim,'train_n':len(ys),'test_n':len(yt),**metric_row(yt,yp)}
            runs.append(row)
            for s,y,p in zip(testseq,yt,yp):preds.append({'task':'MIC_E_coli','split':split,'features':dim,'sequence':s,'observed_log10_uM':y,'predicted_log10_uM':p})
        print('E.coli split',split,'complete, train',len(ys),'test',len(yt))
    except Exception as e:
        errors.append({'task':'MIC_E_coli','split':split,'error':repr(e)})
        print('E.coli split',split,'ERROR:',repr(e)); raise

## HC50 task: QMAP-to-QMAP benchmark

Benchmark uses QMAP's DBAASP source with train masks. Models predict log10 HC50 in µM. It is not an original HC50 model replica because the original PACE model was trained on the GRAMPA-associated public hemolysis corpus.

In [ ]:
dbase_h=canonical_subset(DATASET.with_hc50())
for split in range(5):
    try:
        benchmark=canonical_subset(QMAPBenchmark(split=split).with_hc50())
        trmask=benchmark.get_train_mask(list(dbase_h.sequences),threshold=.6,show_progress=False,num_threads=2)
        train=dbase_h[trmask]
        trseq=list(train.sequences);teseq=list(benchmark.sequences)
        ys=np.array([float(x.hc50.consensus) for x in train],dtype=float)
        yt=np.array([float(x.hc50.consensus) for x in benchmark],dtype=float)
        keep=np.isfinite(ys)&(ys>0);ok=np.isfinite(yt)&(yt>0)
        trseq=[s for s,k in zip(trseq,keep) if k];teseq=[s for s,k in zip(teseq,ok) if k]
        ys=np.log10(ys[keep]);yt=np.log10(yt[ok])
        if len(trseq)<40 or len(teseq)<20: raise RuntimeError('Not enough train/test HC50 samples after filters')
        for dim in (21,90):
            yp=model_fit_predict(trseq,ys,teseq,dim)
            runs.append({'task':'HC50_QMAP_train','split':split,'features':dim,'train_n':len(ys),'test_n':len(yt),**metric_row(yt,yp)})
            for s,y,p in zip(teseq,yt,yp):preds.append({'task':'HC50_QMAP_train','split':split,'features':dim,'sequence':s,'observed_log10_uM':y,'predicted_log10_uM':p})
        print('HC50 QMAP split',split,'complete, train',len(ys),'test',len(yt))
    except Exception as e:
        errors.append({'task':'HC50_QMAP_train','split':split,'error':repr(e)})
        print('HC50 split',split,'ERROR:',repr(e)); raise

## HC50 task: original public PACE hemolysis training source, independently evaluated on QMAP

Original PACE HC50 data are fetched from the source repository; test-set homology masking is applied before model fitting. This is a separate cross-corpus check and cannot by itself eliminate differences in assay conditions between Hemolytik-derived data and QMAP DBAASP data.

In [ ]:
import requests
url='https://raw.githubusercontent.com/zswitten/Antimicrobial-Peptides/master/Hemolysis/Cleaned_hemolytic_data.csv'
rr=requests.get(url,timeout=120);rr.raise_for_status()
rawh=rr.content
(OUT/'source_HC50_sha256.txt').write_text(hashlib.sha256(rawh).hexdigest()+'  Cleaned_hemolytic_data.csv\n')
df=pd.read_csv(io.BytesIO(rawh))
assert {'Sequence','log10_HC50'}.issubset(df.columns)
df['clean_seq']=df['Sequence'].astype(str).str.upper().str.strip()
df=df[df['clean_seq'].map(lambda s:5<=len(s)<=60 and all(c in AASET for c in s))]
df['log10_HC50']=pd.to_numeric(df['log10_HC50'],errors='coerce');df=df.dropna(subset=['log10_HC50'])
source_med=df.groupby('clean_seq',as_index=False)['log10_HC50'].median()
source_seq=source_med['clean_seq'].tolist();source_y=source_med['log10_HC50'].to_numpy(float)
print('Cleaned source HC50 unique peptides',len(source_seq))
for split in range(5):
    try:
        b=canonical_subset(QMAPBenchmark(split=split).with_hc50())
        trmask=b.get_train_mask(source_seq,threshold=.6,show_progress=False,num_threads=2)
        trseq=[s for s,k in zip(source_seq,trmask) if k];ys=source_y[np.asarray(trmask,dtype=bool)]
        teseq=list(b.sequences);yt=np.array([float(x.hc50.consensus) for x in b],dtype=float)
        ok=np.isfinite(yt)&(yt>0);teseq=[s for s,k in zip(teseq,ok) if k];yt=np.log10(yt[ok])
        if len(trseq)<40 or len(teseq)<20:raise RuntimeError('Not enough train/test samples')
        for dim in (21,90):
            yp=model_fit_predict(trseq,ys,teseq,dim)
            runs.append({'task':'HC50_PACE_source','split':split,'features':dim,'train_n':len(trseq),'test_n':len(yt),**metric_row(yt,yp)})
            for s,y,p in zip(teseq,yt,yp):preds.append({'task':'HC50_PACE_source','split':split,'features':dim,'sequence':s,'observed_log10_uM':y,'predicted_log10_uM':p})
        print('HC50 PACE-source split',split,'complete, train',len(trseq),'test',len(yt))
    except Exception as e:
        errors.append({'task':'HC50_PACE_source','split':split,'error':repr(e)})
        print('HC50 PACE-source split',split,'ERROR:',repr(e));raise

In [ ]:
from zipfile import ZIP_DEFLATED
res=pd.DataFrame(runs);pred=pd.DataFrame(preds)
assert len(res)==30, f'Expected 3 tasks × 5 splits × 2 representations; got {len(res)}'
assert res.groupby(['task','features']).size().eq(5).all()
res.to_csv(OUT/'qmap_split_metrics.csv',index=False)
pred.to_csv(OUT/'qmap_oof_predictions.csv',index=False)
(res.groupby(['task','features'])[['r2','rmse','mae','pearson','spearman']].agg(['mean','std'])
 .to_csv(OUT/'qmap_descriptive_summary.csv'))
metadata={'protocol':'Five official QMAP splits, homology training masks threshold 0.60, canonical all-L, free N/C termini, length 5–60; 150-tree ExtraTrees; 90D PACE versus 21D AAC+length; MIC E coli from QMAP; HC50 from QMAP and PACE hemolysis source',
          'critical_limitation':'Subset evaluation is not directly comparable with unrestricted QMAP leaderboard; E. coli not one of six original PACE MIC endpoints; no claim of AMP generator superiority',
          'phase8_input_sha256':hashlib.sha256(zpath.read_bytes()).hexdigest(),
          'pace_feature_source_sha256':hashlib.sha256(code).hexdigest(),
          'errors':errors,'python':platform.python_version(), 'qmap_splits':5,
          'counts':{str(k):int(v) for k,v in res.groupby('task').size().items()}}
(OUT/'manifest.json').write_text(json.dumps(metadata,indent=2))
(OUT/'README.txt').write_text('ANALYSIS ONLY. Do not treat model predictions as experimental MIC/HC50. Do not directly compare subset metrics with unrestricted QMAP leaderboard.\n')
outzip=ROOT/'PACE_Phase11B_QMAP_Results.zip'
with ZipFile(outzip,'w',ZIP_DEFLATED) as z:
 for f in OUT.iterdir():
    if f.is_file():z.write(f,arcname=f.name)
print(res.groupby(['task','features'])[['r2','rmse','mae','pearson']].mean().round(3))
print('ZIP prepared:',outzip, 'size',outzip.stat().st_size)
files.download(str(outzip))